# 학생 문자 시스템 — 제4차 정기평가 실전 생성기

제4차 **실제 성적현황 원본 Excel**, 제3차 성적현황 Excel, `FeedbackInput-A.xlsx`를 직접 읽어 학급별 피드백 TXT를 만듭니다.

- 학번으로 학생을 결합하고 이름·학급을 다시 대조합니다.
- 현재 점수와 여섯 영역 합계, 영역 배점, 중복 학번을 검사합니다.
- 제3차 기록이 있는 학생만 점수 변화 문장을 만듭니다.
- `FeedbackInput-A.xlsx`의 수동 문장은 고쳐 쓰지 않고 그대로 넣습니다.
- `PREVIEW`에서는 검토용 TXT를 만들고, `FINAL`에서는 차수 불일치·테스트 문구를 발견하면 저장 전에 중단합니다.
- TXT는 UTF-8 BOM으로 저장하며 기존 파일을 덮어쓰지 않습니다.

## 가장 빠른 사용법

1. 이 노트북을 성적표 Excel 및 `FeedbackInput-A.xlsx`와 같은 폴더에 둡니다.
2. 아래 설정에서 `BASE_DIR`만 확인합니다.
3. 처음에는 `RUN_MODE = "PREVIEW"`로 모두 실행합니다.
4. 검토가 끝나면 `FeedbackInput-A.xlsx`의 시험차수를 4로 저장하고 테스트 문구를 지운 뒤 `RUN_MODE = "FINAL"`로 실행합니다.


## 1. 설정

파일명을 직접 적지 않아도 폴더 안의 A1 제목을 읽어 제4차·제3차 성적표를 찾습니다. 같은 종류의 파일이 여러 개라면 자동 선택하지 않고 경로를 직접 지정하라고 알려 줍니다.


In [ ]:
from pathlib import Path

# 노트북과 Excel 파일이 같은 폴더라면 그대로 둡니다.
BASE_DIR = Path.cwd()

# 자동 검색을 쓰려면 None. 직접 지정하려면 Path(r"C:\...\파일.xlsx") 형식으로 입력합니다.
CURRENT_RESULT_PATH = None
PREVIOUS_RESULT_PATH = None
FEEDBACK_INPUT_PATH = None

OUTPUT_DIR = BASE_DIR / "Feedback_Result"

# PREVIEW: 불일치·테스트 문구를 경고하고 [검토용] TXT 생성
# FINAL: 불일치·테스트 문구가 있으면 TXT 저장 전 중단
RUN_MODE = "PREVIEW"

TARGET_YEAR = 2026
TARGET_EXAM_NO = 4

# 규정서 권장 기본값: 강점 1개, 취약점 2개. 경계 동률은 함께 출력합니다.
MAX_STRENGTHS = 1
MAX_WEAKNESSES = 2
TARGET_AREA_MESSAGES = 3
INCLUDE_BOUNDARY_TIES = True
MAX_RECOMMENDED_CHARS = 1000


## 2. 생성 엔진

아래 셀은 원본 성적표 구조를 직접 읽습니다. `FeedbackInput-A.xlsx`의 학생 시트 A·B·F열은 수식 결과에 의존하지 않고, 학급 시트의 값을 학급명으로 다시 연결합니다.


In [ ]:
from __future__ import annotations

import re
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from typing import Any

from openpyxl import load_workbook


AREA_ORDER = ["어휘 파악", "핵심 요지 파악", "내용 파악", "문맥 파악", "구조 파악", "추론"]
AREA_MAX = {
    "어휘 파악": Decimal("17"),
    "핵심 요지 파악": Decimal("7"),
    "내용 파악": Decimal("21"),
    "문맥 파악": Decimal("12"),
    "구조 파악": Decimal("12"),
    "추론": Decimal("31"),
}
CLASS_HEADERS = ["시험연도", "시험차수", "학급명", "학급평균", "학급별 안내", "전체안내"]
STUDENT_HEADERS = [
    "시험연도", "시험차수", "학급명", "학생명", "학번", "학급평균",
    "등수/성취도", "시험태도", "평소태도", "추천학습", "마지막 문장",
]
TEST_PATTERNS = [
    re.compile(r"테스트\s*더미"),
    re.compile(r"테스트\s*\d+"),
    re.compile(r"휘바"),
    re.compile(r"우휴휴"),
    re.compile(r"드럽게"),
    re.compile(r"골때"),
    re.compile(r"끊으세요"),
    re.compile(r"오예"),
    re.compile(r"열공하삼"),
    re.compile(r"성공하고\s*싶다"),
]


class FeedbackError(RuntimeError):
    pass


@dataclass
class ReportData:
    path: Path
    year: int
    exam_no: int
    rows: list[dict[str, Any]]


@dataclass
class FeedbackData:
    class_rows: dict[str, dict[str, Any]]
    student_rows: dict[str, dict[str, Any]]
    warnings: list[str]
    test_hits: list[str]


def clean_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).replace("\u00a0", " ").strip()


def is_missing(value: Any) -> bool:
    return clean_text(value) == ""


def parse_decimal(value: Any, context: str) -> Decimal:
    raw = clean_text(value).replace(",", "")
    if not raw:
        raise FeedbackError(f"숫자 결측: {context}, 실제값={value!r}")
    try:
        number = Decimal(raw)
    except (InvalidOperation, ValueError):
        raise FeedbackError(f"숫자로 해석할 수 없음: {context}, 실제값={value!r}")
    if not number.is_finite():
        raise FeedbackError(f"유한한 숫자가 아님: {context}, 실제값={value!r}")
    return number


def format_original(number: Decimal) -> str:
    if number == number.to_integral_value():
        return str(int(number))
    return format(number.normalize(), "f")


def format_derived(number: Decimal, places: str = "0.01") -> str:
    return format_original(number.quantize(Decimal(places), rounding=ROUND_HALF_UP))


def student_id_from_cell(cell) -> str:
    value = cell.value
    if value is None:
        return ""
    if isinstance(value, bool):
        return str(value)
    if isinstance(value, int):
        fmt = clean_text(cell.number_format)
        if fmt and re.fullmatch(r"0+", fmt):
            return f"{value:0{len(fmt)}d}"
        return str(value)
    if isinstance(value, float) and value.is_integer():
        fmt = clean_text(cell.number_format)
        if fmt and re.fullmatch(r"0+", fmt):
            return f"{int(value):0{len(fmt)}d}"
        return str(int(value))
    return clean_text(value)


def parse_metadata(value: Any, path: Path) -> tuple[int, int]:
    text = re.sub(r"\s+", " ", clean_text(value))
    year_match = re.search(r"((?:19|20)\d{2})\s*년?", text)
    exam_match = re.search(r"제\s*(\d+)\s*차", text)
    if not year_match or not exam_match:
        raise FeedbackError(f"A1에서 시험 연도·차수를 찾지 못했습니다: {path.name}, A1={value!r}")
    return int(year_match.group(1)), int(exam_match.group(1))


def excel_files(folder: Path) -> list[Path]:
    return sorted(
        path for path in folder.glob("*.xlsx")
        if not path.name.startswith("~$")
    )


def report_candidates(folder: Path, year: int, exam_no: int) -> list[Path]:
    candidates = []
    for path in excel_files(folder):
        if "FeedbackInput" in path.name:
            continue
        try:
            wb = load_workbook(path, read_only=True, data_only=True)
            ws = wb.active
            detected = parse_metadata(ws["A1"].value, path)
            wb.close()
        except Exception:
            continue
        if detected == (year, exam_no):
            candidates.append(path)
    return candidates


def resolve_one(explicit: Any, candidates: list[Path], label: str) -> Path:
    if explicit is not None:
        path = Path(explicit)
        if not path.exists():
            raise FeedbackError(f"{label} 파일을 찾을 수 없습니다: {path}")
        return path
    if len(candidates) != 1:
        raise FeedbackError(
            f"{label} 자동 검색 결과가 {len(candidates)}개입니다. 설정 셀에 경로를 직접 지정하세요: "
            f"{[path.name for path in candidates]}"
        )
    return candidates[0]


def resolve_paths(
    base_dir: Path,
    current_path: Any = None,
    previous_path: Any = None,
    feedback_path: Any = None,
    year: int = TARGET_YEAR,
    exam_no: int = TARGET_EXAM_NO,
) -> tuple[Path, Path | None, Path]:
    base_dir = Path(base_dir)
    if not base_dir.exists():
        raise FeedbackError(f"BASE_DIR 폴더를 찾을 수 없습니다: {base_dir}")
    current = resolve_one(current_path, report_candidates(base_dir, year, exam_no), "현재 성적표")

    if previous_path is None:
        previous_candidates = report_candidates(base_dir, year, exam_no - 1)
        previous = resolve_one(None, previous_candidates, "이전 성적표") if previous_candidates else None
    else:
        previous = resolve_one(previous_path, [], "이전 성적표")

    if feedback_path is None:
        feedback_candidates = [path for path in excel_files(base_dir) if path.name.startswith("FeedbackInput-A")]
        feedback = resolve_one(None, feedback_candidates, "FeedbackInput-A")
    else:
        feedback = resolve_one(feedback_path, [], "FeedbackInput-A")
    return current, previous, feedback


def find_row(ws, required: set[str], max_rows: int = 10) -> int:
    for row_no in range(1, min(ws.max_row, max_rows) + 1):
        values = {clean_text(ws.cell(row_no, col).value) for col in range(1, ws.max_column + 1)}
        if required.issubset(values):
            return row_no
    raise FeedbackError(f"시트 '{ws.title}'에서 헤더 {sorted(required)}를 찾지 못했습니다.")


def first_column(ws, row_no: int, name: str) -> int:
    columns = [col for col in range(1, ws.max_column + 1) if clean_text(ws.cell(row_no, col).value) == name]
    if not columns:
        raise FeedbackError(f"시트 '{ws.title}' {row_no}행에 '{name}' 열이 없습니다.")
    return columns[0]


def read_current_report(path: Path, expected_year: int, expected_exam_no: int) -> ReportData:
    wb = load_workbook(path, read_only=False, data_only=True)
    ws = wb.active
    year, exam_no = parse_metadata(ws["A1"].value, path)
    if (year, exam_no) != (expected_year, expected_exam_no):
        raise FeedbackError(
            f"현재 성적표의 시험 정보가 다릅니다: 기대={expected_year}년 제{expected_exam_no}차, "
            f"실제={year}년 제{exam_no}차, 파일={path.name}"
        )

    header_row = find_row(ws, {"학급명", "학생명", "학번", "점수"})
    area_row = header_row + 1
    columns = {
        "학급명": first_column(ws, header_row, "학급명"),
        "학생명": first_column(ws, header_row, "학생명"),
        "학번": first_column(ws, header_row, "학번"),
        "점수": first_column(ws, header_row, "점수"),
    }
    area_columns = {area: first_column(ws, area_row, area) for area in AREA_ORDER}
    duplicate_area_columns = {
        area: [col for col in range(1, ws.max_column + 1) if clean_text(ws.cell(area_row, col).value) == area]
        for area in AREA_ORDER
    }

    rows = []
    seen_ids = set()
    for excel_row in range(area_row + 1, ws.max_row + 1):
        class_name = clean_text(ws.cell(excel_row, columns["학급명"]).value)
        name = clean_text(ws.cell(excel_row, columns["학생명"]).value)
        sid = student_id_from_cell(ws.cell(excel_row, columns["학번"]))
        score_raw = ws.cell(excel_row, columns["점수"]).value
        if not any((class_name, name, sid, clean_text(score_raw))):
            continue
        if not class_name or not name or not sid:
            raise FeedbackError(
                f"필수값 결측: 파일={path.name}, Excel행={excel_row}, "
                f"학급명={class_name!r}, 학생명={name!r}, 학번={sid!r}"
            )
        if sid in seen_ids:
            raise FeedbackError(f"동일 학번 중복: 파일={path.name}, 학번={sid}, Excel행={excel_row}")
        seen_ids.add(sid)

        absent = is_missing(score_raw)
        record = {
            "학급명": class_name,
            "학생명": name,
            "학번": sid,
            "결시": absent,
            "점수": None,
            "영역": {},
            "_excel_row": excel_row,
        }
        if absent:
            rows.append(record)
            continue

        current = parse_decimal(score_raw, f"파일={path.name}, Excel행={excel_row}, 학생={name}, 열=점수")
        if current < 0 or current > 100:
            raise FeedbackError(f"현재 점수가 0~100 범위를 벗어났습니다: 학생={name}, 학번={sid}, 값={score_raw!r}")
        record["점수"] = current

        for area in AREA_ORDER:
            primary_col = area_columns[area]
            area_raw = ws.cell(excel_row, primary_col).value
            score = parse_decimal(
                area_raw,
                f"파일={path.name}, Excel행={excel_row}, 학생={name}, 학번={sid}, 열={area}",
            )
            maximum = AREA_MAX[area]
            if score < 0 or score > maximum:
                raise FeedbackError(
                    f"영역 점수가 배점 범위를 벗어났습니다: 학생={name}, 학번={sid}, "
                    f"영역={area}, 값={area_raw!r}, 배점=0~{format_original(maximum)}"
                )
            for duplicate_col in duplicate_area_columns[area][1:]:
                duplicate_raw = ws.cell(excel_row, duplicate_col).value
                if not is_missing(duplicate_raw):
                    duplicate_score = parse_decimal(
                        duplicate_raw,
                        f"파일={path.name}, Excel행={excel_row}, 학생={name}, 중복 영역={area}",
                    )
                    if duplicate_score != score:
                        raise FeedbackError(
                            f"성적표의 중복 영역 점수가 서로 다릅니다: 학생={name}, 학번={sid}, "
                            f"영역={area}, 첫값={score}, 중복값={duplicate_score}"
                        )
            record["영역"][area] = score

        area_total = sum(record["영역"].values(), Decimal("0"))
        tolerance = Decimal("0.01") if any(value != value.to_integral_value() for value in [current, *record["영역"].values()]) else Decimal("0")
        if abs(area_total - current) > tolerance:
            raise FeedbackError(
                f"점수와 여섯 영역 합계가 다릅니다: 파일={path.name}, Excel행={excel_row}, "
                f"학생={name}, 학번={sid}, 점수={current}, 영역합계={area_total}"
            )
        rows.append(record)

    if not rows:
        raise FeedbackError(f"현재 성적표에 학생 데이터가 없습니다: {path.name}")
    wb.close()
    return ReportData(path, year, exam_no, rows)


def read_previous_scores(path: Path | None, expected_year: int, expected_exam_no: int) -> tuple[dict[str, dict[str, Any]], list[str]]:
    if path is None:
        return {}, ["이전 성적표가 없어 모든 학생의 점수 변화 문장을 생략합니다."]
    wb = load_workbook(path, read_only=False, data_only=True)
    ws = wb.active
    year, exam_no = parse_metadata(ws["A1"].value, path)
    if (year, exam_no) != (expected_year, expected_exam_no):
        raise FeedbackError(
            f"이전 성적표의 시험 정보가 다릅니다: 기대={expected_year}년 제{expected_exam_no}차, "
            f"실제={year}년 제{exam_no}차, 파일={path.name}"
        )
    header_row = find_row(ws, {"학급명", "학생명", "학번"})
    id_col = first_column(ws, header_row, "학번")
    name_col = first_column(ws, header_row, "학생명")
    score_label = f"{expected_exam_no}차"
    try:
        score_col = first_column(ws, header_row, score_label)
    except FeedbackError:
        score_col = first_column(ws, header_row, "점수")
    result = {}
    for excel_row in range(header_row + 2, ws.max_row + 1):
        sid = student_id_from_cell(ws.cell(excel_row, id_col))
        name = clean_text(ws.cell(excel_row, name_col).value)
        if not sid and not name:
            continue
        if not sid or not name:
            raise FeedbackError(f"이전 성적표 필수값 결측: 파일={path.name}, Excel행={excel_row}")
        if sid in result:
            raise FeedbackError(f"이전 성적표 동일 학번 중복: 파일={path.name}, 학번={sid}")
        score_raw = ws.cell(excel_row, score_col).value
        result[sid] = {
            "학생명": name,
            "점수": None if is_missing(score_raw) else parse_decimal(
                score_raw,
                f"파일={path.name}, Excel행={excel_row}, 학생={name}, 열={score_label}",
            ),
        }
    wb.close()
    return result, []


def exact_headers(ws, expected: list[str]) -> None:
    actual = [clean_text(ws.cell(1, col).value) for col in range(1, ws.max_column + 1)]
    while actual and actual[-1] == "":
        actual.pop()
    if actual != expected:
        raise FeedbackError(
            f"FeedbackInput-A 시트 '{ws.title}' 열이 규정과 다릅니다.\n기대={expected}\n실제={actual}"
        )


def find_test_hits(sheet_name: str, row_no: int, values: dict[str, str]) -> list[str]:
    hits = []
    for column, value in values.items():
        if any(pattern.search(value) for pattern in TEST_PATTERNS):
            hits.append(f"{sheet_name} {row_no}행 '{column}': {value[:80]}")
    return hits


def load_feedback_input(
    path: Path,
    current: ReportData,
    mode: str,
) -> FeedbackData:
    wb = load_workbook(path, read_only=False, data_only=False)
    required_sheets = {"학급메모", "학생메모"}
    if set(wb.sheetnames) != required_sheets:
        raise FeedbackError(
            f"FeedbackInput-A 시트는 정확히 {sorted(required_sheets)}여야 합니다: 실제={wb.sheetnames}"
        )
    class_ws = wb["학급메모"]
    student_ws = wb["학생메모"]
    exact_headers(class_ws, CLASS_HEADERS)
    exact_headers(student_ws, STUDENT_HEADERS)

    mode = mode.upper()
    warnings = []
    test_hits = []
    current_classes = {row["학급명"] for row in current.rows}
    current_students = {row["학번"]: row for row in current.rows}

    class_rows = {}
    metadata_mismatches = []
    for row_no in range(2, class_ws.max_row + 1):
        values = {header: class_ws.cell(row_no, index + 1).value for index, header in enumerate(CLASS_HEADERS)}
        if all(is_missing(value) for value in values.values()):
            continue
        class_name = clean_text(values["학급명"])
        if not class_name:
            raise FeedbackError(f"FeedbackInput-A 학급명 결측: 학급메모 {row_no}행")
        if class_name in class_rows:
            raise FeedbackError(f"FeedbackInput-A 학급 중복: 학급메모 {row_no}행, 학급명={class_name}")
        year = int(parse_decimal(values["시험연도"], f"학급메모 {row_no}행 시험연도"))
        exam_no = int(parse_decimal(values["시험차수"], f"학급메모 {row_no}행 시험차수"))
        if (year, exam_no) != (current.year, current.exam_no):
            metadata_mismatches.append(
                f"학급메모 {row_no}행 {class_name}: 입력={year}년 제{exam_no}차, "
                f"성적표={current.year}년 제{current.exam_no}차"
            )
        average = parse_decimal(values["학급평균"], f"학급메모 {row_no}행 학급평균")
        manual = {
            "학급별 안내": clean_text(values["학급별 안내"]),
            "전체안내": clean_text(values["전체안내"]),
        }
        test_hits.extend(find_test_hits("학급메모", row_no, manual))
        class_rows[class_name] = {"시험연도": year, "시험차수": exam_no, "학급평균": average, **manual}

    missing_classes = current_classes - set(class_rows)
    extra_classes = set(class_rows) - current_classes
    if missing_classes or extra_classes:
        raise FeedbackError(
            f"FeedbackInput-A와 현재 성적표의 학급 구성이 다릅니다: "
            f"누락={sorted(missing_classes)}, 추가={sorted(extra_classes)}"
        )

    student_rows = {}
    for row_no in range(2, student_ws.max_row + 1):
        class_name = clean_text(student_ws.cell(row_no, 3).value)
        name = clean_text(student_ws.cell(row_no, 4).value)
        sid = student_id_from_cell(student_ws.cell(row_no, 5))
        manual = {
            "등수/성취도": clean_text(student_ws.cell(row_no, 7).value),
            "시험태도": clean_text(student_ws.cell(row_no, 8).value),
            "평소태도": clean_text(student_ws.cell(row_no, 9).value),
            "추천학습": clean_text(student_ws.cell(row_no, 10).value),
            "마지막 문장": clean_text(student_ws.cell(row_no, 11).value),
        }
        if not any((class_name, name, sid, *manual.values())):
            continue
        if not class_name or not name or not sid:
            raise FeedbackError(
                f"FeedbackInput-A 학생 필수값 결측: 학생메모 {row_no}행, "
                f"학급명={class_name!r}, 학생명={name!r}, 학번={sid!r}"
            )
        if sid in student_rows:
            raise FeedbackError(f"FeedbackInput-A 학생 학번 중복: 학생메모 {row_no}행, 학번={sid}")
        if sid not in current_students:
            raise FeedbackError(f"현재 성적표에 없는 학생: 학생메모 {row_no}행, 학생={name}, 학번={sid}")
        source = current_students[sid]
        if source["학생명"] != name or source["학급명"] != class_name:
            raise FeedbackError(
                f"학번의 학생명·학급명이 일치하지 않습니다: 학번={sid}, "
                f"성적표=({source['학급명']}, {source['학생명']}), FeedbackInput=({class_name}, {name})"
            )
        test_hits.extend(find_test_hits("학생메모", row_no, manual))
        student_rows[sid] = {"학급명": class_name, "학생명": name, **manual}

    missing_students = set(current_students) - set(student_rows)
    extra_students = set(student_rows) - set(current_students)
    if missing_students or extra_students:
        raise FeedbackError(
            f"FeedbackInput-A와 현재 성적표의 학생 구성이 다릅니다: "
            f"누락학번={sorted(missing_students)}, 추가학번={sorted(extra_students)}"
        )

    if metadata_mismatches:
        message = "FeedbackInput-A 시험 정보 불일치: " + " | ".join(metadata_mismatches)
        if mode == "FINAL":
            raise FeedbackError(message)
        warnings.append(message)
    if test_hits:
        message = f"검토가 필요한 테스트·비속어 표식이 {len(test_hits)}개 있습니다."
        if mode == "FINAL":
            raise FeedbackError(message + "\n- " + "\n- ".join(test_hits))
        warnings.append(message)
    wb.close()
    return FeedbackData(class_rows, student_rows, warnings, test_hits)


def has_final_consonant(text: str) -> bool:
    if not text:
        raise FeedbackError("학생 이름이 비어 있습니다.")
    code = ord(text[-1])
    if not (0xAC00 <= code <= 0xD7A3):
        raise FeedbackError(f"한글로 끝나지 않는 학생명은 처리할 수 없습니다: {text!r}")
    return (code - 0xAC00) % 28 != 0


def friendly_subject(full_name: str) -> str:
    name = re.sub(r"\s+", "", clean_text(full_name))[1:]
    if not name:
        raise FeedbackError(f"성을 제외한 이름을 만들 수 없습니다: {full_name!r}")
    return name + ("이는" if has_final_consonant(name) else "는")


AREA_MESSAGES = {
    "어휘 파악": {
        "zero": "‘어휘’가 0점입니다. 문해력의 기초가 되는 어휘가 안정되어야 더 어려운 영역도 순차적으로 향상될 수 있습니다. 과제를 할 때 어휘의 뜻을 자신이 아는 쉬운 말로 바꾸어 익히는 연습이 도움이 될 것입니다.",
        "perfect": "‘어휘’가 만점으로 문해력의 기초가 되는 어휘를 잘 익히고 있는 것으로 보입니다.",
        "weak": "‘어휘’의 득점률은 {rate}%입니다. 어휘는 문해력의 기초가 되는 부분으로, 뜻을 자신이 아는 쉬운 말로 바꾸어 익히는 연습이 도움이 될 것입니다.",
        "strong": "‘어휘’의 득점률은 {rate}%로 문해력의 기초가 되는 어휘를 잘 익히고 있는 것으로 보입니다.",
        "general": "",
    },
    "핵심 요지 파악": {
        "zero": "‘핵심 요지 파악’이 0점입니다. 글 전체를 읽고 주제를 고르는 영역은 글의 일부분만으로 답을 찾기 어려워 꾸준한 훈련이 필요합니다.",
        "perfect": "‘핵심 요지 파악’이 만점입니다. 글 전체를 종합해 주제를 찾는 영역에서 어려운 지문의 내용을 잘 이해한 결과입니다.",
        "weak": "‘핵심 요지 파악’의 득점률은 {rate}%입니다. 글 전체의 내용을 종합해 주제를 고르는 연습이 필요합니다.",
        "strong": "‘핵심 요지 파악’의 득점률은 {rate}%입니다. 글 전체를 종합해 주제를 찾는 영역에서 어려운 지문의 내용을 잘 이해한 결과입니다.",
        "general": "‘핵심 요지 파악’은 글 전체의 내용을 종합해 주제를 고르는 연습이 필요한 영역입니다.",
    },
    "내용 파악": {
        "zero": "‘내용 파악’이 0점입니다. 지문과 선택지를 세세하게 대조하는 훈련이 필요합니다.",
        "perfect": "‘내용 파악’이 만점입니다. 지문과 선택지를 세세하게 대조하며 어려운 지문의 내용을 빠르고 정확하게 이해한 결과입니다.",
        "weak": "‘내용 파악’의 득점률은 {rate}%입니다. 지문과 선택지를 세세하게 대조하는 훈련이 필요합니다.",
        "strong": "‘내용 파악’의 득점률은 {rate}%입니다. 지문과 선택지를 세세하게 대조하며 어려운 지문의 내용을 빠르고 정확하게 이해한 결과입니다.",
        "general": "‘내용 파악’은 지문과 선택지를 세세하게 대조하는 훈련이 필요한 영역입니다.",
    },
    "문맥 파악": {
        "zero": "‘문맥 파악’이 0점입니다. 앞뒤 문장의 내용을 함께 살펴 의미를 판단하는 꾸준한 훈련이 필요합니다.",
        "perfect": "‘문맥 파악’이 만점입니다. 앞뒤 문장의 내용을 정확하게 연결해 어려운 지문의 내용을 잘 이해한 결과입니다.",
        "weak": "‘문맥 파악’의 득점률은 {rate}%입니다. 앞뒤 문장의 내용을 함께 살펴 의미를 판단하는 꾸준한 훈련이 필요합니다.",
        "strong": "‘문맥 파악’의 득점률은 {rate}%입니다. 앞뒤 문장의 내용을 정확하게 연결해 어려운 지문의 내용을 잘 이해한 결과입니다.",
        "general": "‘문맥 파악’은 앞뒤 문장의 내용을 함께 살펴 의미를 판단하는 훈련이 필요한 영역입니다.",
    },
    "구조 파악": {
        "zero": "‘구조 파악’이 0점입니다. 앞뒤 문장과 접속사의 관계를 살피며 문단의 적절한 위치를 찾는 기술적인 훈련이 필요합니다.",
        "perfect": "‘구조 파악’이 만점입니다. 앞뒤 문장과 접속사의 관계를 파악해 문단의 위치를 찾는 기술적인 훈련이 잘되어 있습니다.",
        "weak": "‘구조 파악’의 득점률은 {rate}%입니다. 앞뒤 문장과 접속사의 관계를 살피며 문단의 적절한 위치를 찾는 기술적인 훈련이 필요합니다.",
        "strong": "‘구조 파악’의 득점률은 {rate}%입니다. 앞뒤 문장과 접속사의 관계를 파악해 문단의 위치를 찾는 기술적인 훈련이 잘되어 있습니다.",
        "general": "",
    },
    "추론": {
        "zero": "‘추론’이 0점입니다. 지문에 직접 드러나지 않은 내용을 문맥을 바탕으로 추리하는 영역으로 꾸준한 훈련이 필요합니다.",
        "perfect": "‘추론’이 만점입니다. 지문에 직접 드러나지 않은 내용을 문맥을 바탕으로 정확하게 판단한 결과입니다.",
        "weak": "‘추론’의 득점률은 {rate}%입니다. 지문에 직접 드러나지 않은 내용을 문맥을 바탕으로 추리하는 훈련이 필요합니다.",
        "strong": "‘추론’의 득점률은 {rate}%입니다. 지문에 직접 드러나지 않은 내용을 문맥을 바탕으로 정확하게 판단한 결과입니다.",
        "general": "‘추론’은 지문에 직접 드러나지 않은 내용을 문맥을 바탕으로 판단하는 훈련이 필요한 영역입니다.",
    },
}


def area_items(area_scores: dict[str, Decimal]) -> list[dict[str, Any]]:
    items = []
    for index, area in enumerate(AREA_ORDER):
        score = area_scores[area]
        maximum = AREA_MAX[area]
        rate = score / maximum * Decimal("100")
        if score == maximum:
            state = "perfect"
        elif score == 0:
            state = "zero"
        elif rate >= 80:
            state = "strong"
        elif rate <= 50:
            state = "weak"
        else:
            state = "general"
        items.append({"name": area, "index": index, "score": score, "max": maximum, "rate": rate, "state": state})
    return items


def render_area(item: dict[str, Any], state: str | None = None) -> str:
    selected = state or item["state"]
    return AREA_MESSAGES[item["name"]][selected].format(rate=format_derived(item["rate"]))


def select_ties(items: list[dict[str, Any]], limit: int, descending: bool) -> list[dict[str, Any]]:
    ordered = sorted(items, key=lambda item: ((-item["rate"] if descending else item["rate"]), item["index"]))
    if len(ordered) <= limit or not INCLUDE_BOUNDARY_TIES:
        return ordered[:limit]
    boundary = ordered[limit - 1]["rate"]
    return [item for item in ordered if (item["rate"] >= boundary if descending else item["rate"] <= boundary)]


def build_area_blocks(area_scores: dict[str, Decimal]) -> list[str]:
    items = area_items(area_scores)
    strengths = select_ties([item for item in items if item["state"] in {"perfect", "strong"}], MAX_STRENGTHS, True)
    weak_candidates = [item for item in items if item["state"] in {"zero", "weak"}]

    if len(weak_candidates) >= 4:
        ordered_weak = sorted(weak_candidates, key=lambda item: (item["rate"], item["index"]))
        error_rates = [Decimal("100") - item["rate"] for item in weak_candidates]
        broad = (
            "영역별로 보면 여러 영역에서 고르게 오답이 확인되었습니다. "
            f"영역별 오답률은 {format_derived(min(error_rates))}%부터 {format_derived(max(error_rates))}%까지 분포해 "
            "전체 영역을 꾸준히 공부할 필요가 있습니다."
        )
        general_details = [render_area(item, "general") for item in ordered_weak[:3] if AREA_MESSAGES[item["name"]]["general"]]
        return [broad, *general_details]

    weaknesses = select_ties(weak_candidates, MAX_WEAKNESSES, False)
    blocks = [render_area(item) for item in strengths] + [render_area(item) for item in weaknesses]
    if len(blocks) < TARGET_AREA_MESSAGES:
        general_candidates = [
            item for item in items
            if item["state"] == "general" and AREA_MESSAGES[item["name"]]["general"]
        ]
        for item in general_candidates:
            if len(blocks) >= TARGET_AREA_MESSAGES:
                break
            blocks.append(render_area(item, "general"))
    return blocks


def unique_blocks(blocks: list[str]) -> list[str]:
    result = []
    seen = set()
    for block in blocks:
        value = clean_text(block)
        if not value:
            continue
        key = re.sub(r"\s+", " ", value)
        if key not in seen:
            seen.add(key)
            result.append(value)
    return result


def class_top_ids(rows: list[dict[str, Any]]) -> set[str]:
    scores = {row["학번"]: row["점수"] for row in rows if not row["결시"]}
    if not scores:
        return set()
    maximum = max(scores.values())
    return {sid for sid, score in scores.items() if score == maximum}


def standard_change(current: Decimal, previous: Decimal, previous_exam_no: int) -> str:
    difference = current - previous
    if difference == 0:
        return f"지난 {previous_exam_no}차 정기평가와 같은 점수를 받았습니다."
    magnitude = format_original(abs(difference))
    if difference > 0:
        return f"지난 {previous_exam_no}차 정기평가 점수와 비교해 보면 {magnitude}점 향상되었습니다."
    return f"지난 {previous_exam_no}차 정기평가 점수와 비교해 보면 {magnitude}점 낮아졌습니다."


def achievement_change(current: Decimal, previous: Decimal, previous_exam_no: int) -> str:
    difference = current - previous
    if difference == 0:
        return f"지난 {previous_exam_no}차 정기평가와 같은 점수로,"
    magnitude = format_original(abs(difference))
    if difference > 0:
        return f"지난 {previous_exam_no}차 정기평가 점수와 비교해 보면 {magnitude}점 향상된 점수로,"
    return f"지난 {previous_exam_no}차 정기평가 점수와 비교해 보면 {magnitude}점 낮은 점수이지만,"


def score_block(row: dict[str, Any], previous: Decimal | None, manual_achievement: str, top_ids: set[str], top_count: int) -> str:
    subject = friendly_subject(row["학생명"])
    current = row["점수"]
    achievements = []
    compact_manual = re.sub(r"\s+", "", manual_achievement)
    if row["학번"] in top_ids and "1등" not in compact_manual:
        achievements.append("우리 반 1등입니다." if top_count == 1 else "우리 반 공동 1등입니다.")
    if manual_achievement:
        achievements.append(manual_achievement)

    if achievements:
        first = f"{subject} 이번 시험에서 {format_original(current)}점을 받았습니다."
        if previous is None:
            return " ".join([first, *achievements])
        return " ".join([first, achievement_change(current, previous, TARGET_EXAM_NO - 1), *achievements])

    first = f"{subject} 이번 정기평가에서 {format_original(current)}점을 받았습니다."
    if previous is None:
        return first
    return f"{subject} 이번 정기평가에서 {format_original(current)}점을 받았으며, {standard_change(current, previous, TARGET_EXAM_NO - 1)}"


def computed_class_average(rows: list[dict[str, Any]]) -> Decimal | None:
    scores = [row["점수"] for row in rows if not row["결시"]]
    return sum(scores, Decimal("0")) / Decimal(len(scores)) if scores else None


def remove_teacher_name(class_name: str) -> str:
    return re.sub(r"-[^-]+T$", "", class_name).strip("-")


def sanitize_filename(name: str) -> str:
    return re.sub(r'[<>:"/\\|?*]', "_", name).rstrip(". ")


def choose_path(output_dir: Path, year: int, exam_no: int, class_name: str, mode: str) -> Path:
    prefix = "[검토용] " if mode.upper() == "PREVIEW" else ""
    name = sanitize_filename(f"{prefix}[{year}-{exam_no}차] {remove_teacher_name(class_name)}-피드백.txt")
    target = output_dir / name
    if target.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        target = output_dir / f"{target.stem}_{stamp}{target.suffix}"
    return target


def render_txt(messages: list[tuple[str, str, str]]) -> str:
    sections = [f"▷ {name}\n\n{message}" for name, _, message in messages]
    return "\n\n--------------------------\n\n".join(sections) + "\n"


def build_messages(
    current: ReportData,
    previous_scores: dict[str, dict[str, Any]],
    feedback: FeedbackData,
) -> tuple[dict[str, list[tuple[str, str, str]]], list[str], dict[str, int]]:
    warnings = list(feedback.warnings)
    grouped = defaultdict(list)
    for row in current.rows:
        grouped[row["학급명"]].append(row)

    outputs = {}
    counts = {"students": 0, "absent": 0, "previous_missing": 0, "over_length": 0}
    for class_name in sorted(grouped):
        rows = grouped[class_name]
        class_input = feedback.class_rows[class_name]
        calculated_average = computed_class_average(rows)
        supplied_average = class_input["학급평균"]
        if calculated_average is not None and abs(calculated_average - supplied_average) > Decimal("0.01"):
            warnings.append(
                f"학급평균 대조 경고: {class_name}, FeedbackInput={format_original(supplied_average)}, "
                f"성적표 계산={format_derived(calculated_average)}. 규정에 따라 FeedbackInput 값을 사용합니다."
            )
        top_ids = class_top_ids(rows)
        messages = []
        for row in sorted(rows, key=lambda item: (item["학생명"], item["학번"])):
            counts["students"] += 1
            name = row["학생명"]
            sid = row["학번"]
            if row["결시"]:
                counts["absent"] += 1
                messages.append((name, sid, f"{friendly_subject(name)} ★★★ 시험 미실시 또는 중도 퇴원 ★★★"))
                continue

            previous = previous_scores.get(sid)
            previous_score = None
            if previous is not None:
                if previous["학생명"] != name:
                    raise FeedbackError(
                        f"이전 성적표와 학생명이 다릅니다: 학번={sid}, 현재={name}, 이전={previous['학생명']}"
                    )
                previous_score = previous["점수"]
            if previous_score is None:
                counts["previous_missing"] += 1

            student_input = feedback.student_rows[sid]
            blocks = [
                f"우리 반 평균은 {format_original(supplied_average)}점입니다.\n------------------------",
                score_block(
                    row,
                    previous_score,
                    student_input["등수/성취도"],
                    top_ids,
                    len(top_ids),
                ),
                class_input["전체안내"],
                class_input["학급별 안내"],
                *build_area_blocks(row["영역"]),
                student_input["시험태도"],
                student_input["평소태도"],
                student_input["추천학습"],
                student_input["마지막 문장"],
            ]
            message = "\n\n".join(unique_blocks(blocks))
            if len(message) > MAX_RECOMMENDED_CHARS:
                counts["over_length"] += 1
                warnings.append(
                    f"권장 글자 수 초과: 학급={class_name}, 학생={name}, 학번={sid}, "
                    f"글자수={len(message)}, 권장={MAX_RECOMMENDED_CHARS}"
                )
            messages.append((name, sid, message))
        outputs[class_name] = messages
    return outputs, warnings, counts


def run_feedback_generation(
    base_dir: Path = BASE_DIR,
    current_path: Any = CURRENT_RESULT_PATH,
    previous_path: Any = PREVIOUS_RESULT_PATH,
    feedback_path: Any = FEEDBACK_INPUT_PATH,
    output_dir: Path = OUTPUT_DIR,
    mode: str = RUN_MODE,
) -> dict[str, Any]:
    mode = clean_text(mode).upper()
    if mode not in {"PREVIEW", "FINAL"}:
        raise FeedbackError("RUN_MODE는 'PREVIEW' 또는 'FINAL'이어야 합니다.")
    current_path, previous_path, feedback_path = resolve_paths(
        Path(base_dir), current_path, previous_path, feedback_path, TARGET_YEAR, TARGET_EXAM_NO
    )
    current = read_current_report(current_path, TARGET_YEAR, TARGET_EXAM_NO)
    previous_scores, previous_warnings = read_previous_scores(previous_path, TARGET_YEAR, TARGET_EXAM_NO - 1)
    feedback = load_feedback_input(feedback_path, current, mode)
    messages_by_class, warnings, counts = build_messages(current, previous_scores, feedback)
    warnings = [*previous_warnings, *warnings]

    # 모든 검증과 문구 조립이 끝난 뒤에만 파일을 저장합니다.
    output_dir = Path(output_dir)
    prepared = []
    names = set()
    for class_name, messages in messages_by_class.items():
        target = choose_path(output_dir, current.year, current.exam_no, class_name, mode)
        if target.name in names:
            raise FeedbackError(f"출력 파일명이 충돌합니다: {target.name}")
        names.add(target.name)
        prepared.append((target, render_txt(messages)))

    output_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    for target, content in prepared:
        temporary = target.with_suffix(target.suffix + ".tmp")
        temporary.write_text(content, encoding="utf-8-sig", newline="\n")
        temporary.replace(target)
        saved.append(target)

    print(f"실행 모드: {mode}")
    print(f"현재 성적표: {current_path.name}")
    print(f"이전 성적표: {previous_path.name if previous_path else '없음'}")
    print(f"FeedbackInput-A: {feedback_path.name}")
    print(f"처리 학급: {len(messages_by_class)}개")
    print(f"처리 학생: {counts['students']}명")
    print(f"결시: {counts['absent']}명")
    print(f"이전점수 없음: {counts['previous_missing']}명")
    print(f"1000자 초과: {counts['over_length']}명")
    print(f"테스트 표식: {len(feedback.test_hits)}개")
    print("저장 파일:")
    for path in saved:
        print(f"- {path}")
    if warnings:
        print("\n[경고]")
        for warning in warnings:
            print(f"- {warning}")
    if feedback.test_hits:
        print("\n[테스트 표식 위치]")
        for hit in feedback.test_hits:
            print(f"- {hit}")

    return {
        "mode": mode,
        "year": current.year,
        "exam_no": current.exam_no,
        "class_count": len(messages_by_class),
        **counts,
        "test_hits": feedback.test_hits,
        "warnings": warnings,
        "saved": saved,
    }


## 3. 내장 테스트

실제 파일을 저장하기 전에 이름 조사, 점수 변화 문장, 영역 합계와 파일명 규칙을 확인합니다.


In [ ]:
def run_self_tests() -> None:
    assert friendly_subject("김주원") == "주원이는"
    assert friendly_subject("박지우") == "지우는"
    assert format_original(Decimal("48.0")) == "48"
    assert format_original(Decimal("48.5")) == "48.5"
    assert standard_change(Decimal("70"), Decimal("60"), 3) == "지난 3차 정기평가 점수와 비교해 보면 10점 향상되었습니다."
    assert standard_change(Decimal("60"), Decimal("70"), 3) == "지난 3차 정기평가 점수와 비교해 보면 10점 낮아졌습니다."
    assert standard_change(Decimal("60"), Decimal("60"), 3) == "지난 3차 정기평가와 같은 점수를 받았습니다."
    sample_areas = {
        "어휘 파악": Decimal("17"),
        "핵심 요지 파악": Decimal("7"),
        "내용 파악": Decimal("18"),
        "문맥 파악": Decimal("9"),
        "구조 파악": Decimal("9"),
        "추론": Decimal("20"),
    }
    assert sum(sample_areas.values(), Decimal("0")) == Decimal("80")
    assert build_area_blocks(sample_areas)
    assert remove_teacher_name("화랑-금1830-임서영T") == "화랑-금1830"
    print("내장 테스트 통과")


run_self_tests()


## 4. 실제 TXT 생성

처음에는 반드시 `PREVIEW`로 실행해 문장을 확인하세요. `FINAL`은 `FeedbackInput-A.xlsx`의 시험차수가 제4차이고 테스트 표식이 없을 때만 저장됩니다.


In [ ]:
result = run_feedback_generation()
result
